In [ ]:
!pip install anyascii

import os
import pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from anyascii import anyascii

# 1. KHỞI TẠO SPARK VỚI CẤU HÌNH BẬT TỐC ĐỘ ARROW (VECTORIZED)
spark = SparkSession.builder \
    .appName("Clean_All_Files_ULTIMATE") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# Pandas UDF dịch chữ Latin
@F.pandas_udf(StringType())
def translit_pandas_udf(s: pd.Series) -> pd.Series:
    return s.apply(lambda x: anyascii(x) if pd.notnull(x) else None)

# 2. KHAI BÁO THƯ MỤC
# Lưu ý: Nhớ thay 2 cái THUMUC_1, THUMUC_2 bằng tên thư mục thật của mày
folder1_path = "/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/CLEARDATA"
folder2_path = "/kaggle/input/datasets/js042710/second3t1k/data DM/data DM"
output_dir = "/kaggle/working/CLEANED_DATASET_PARQUET"

# SỬA LỖI ĐƯỜNG DẪN KAGGLE BẰNG SCHEME FILE://
path_1 = f"file://{folder1_path}/*.csv"
path_2 = f"file://{folder2_path}/*.csv"

# 3. ĐỌC SONG SONG MAX TỐC ĐỘ
print("⏳ Đang nạp dữ liệu siêu tốc...")
df = spark.read.csv([path_1, path_2], header=True, inferSchema=False, quote='"', escape='"', multiLine=True)

original_columns = df.columns
sep_pattern = r"\s*[*&/;|,]\s*|\s+(?:x|and|of|feat\.?|ft\.?|with|e)\b\s*"

# 4. CHUỖI LỆNH LÀM SẠCH BẰNG PANDAS UDF
print("⏳ Đang xử lý song song các luồng dữ liệu...")
final_df = df.withColumn("n", F.lower(F.trim(F.col("artist_name")))) \
    .withColumn("n", F.regexp_replace("n", r"\s*\(.*?\)\s*|\s*\[.*?\]\s*", " ")) \
    .withColumn("n", translit_pandas_udf(F.col("n"))) \
    .withColumn("n", F.regexp_replace("n", sep_pattern, "###")) \
    .withColumn("arr", F.split("n", "###")) \
    .withColumn("art", F.explode("arr")) \
    .withColumn("polished", F.regexp_replace(F.trim(F.col("art")), r"^[^a-z0-9]+|[^a-z0-9]+$", "")) \
    .withColumn("artist_name_clean", F.trim(F.col("polished"))) \
    .filter("length(artist_name_clean) >= 2 AND length(artist_name_clean) <= 40") \
    .withColumn("track_name", translit_pandas_udf(F.col("track_name"))) \
    .withColumn("release_name", translit_pandas_udf(F.col("release_name"))) \
    .drop("artist_name") \
    .withColumnRenamed("artist_name_clean", "artist_name") \
    .select(*original_columns)

# 5. GHI PARQUET MỞ CỬA TỰ DO
print("⏳ Đang nén thành Parquet (Spark tự động chia file)...")
final_df.write.mode("overwrite").parquet(output_dir)

print("🎉 DONE! ĐÃ SẠCH SẼ LỖI!")

In [ ]:
# ========================================================
# BỘ CÔNG CỤ NGHIỆM THU DATASET PARQUET
# ========================================================
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()

# 1. Đọc lại Dataset Parquet vừa tạo (Spark đọc cái này chỉ mất vài giây)
parquet_path = "/kaggle/working/CLEANED_DATASET_PARQUET"
print("⏳ Đang nạp lại Dataset để kiểm tra...")
check_df = spark.read.parquet(parquet_path)

# 2. KIỂM TRA TỔNG QUAN (Schema và Số lượng)
print("\n🔍 1. CẤU TRÚC CỘT (SCHEMA):")
check_df.printSchema() # Xem có đủ các cột gốc không, kiểu dữ liệu có đúng không

print("\n🔍 2. TỔNG SỐ DÒNG SAU KHI LÀM SẠCH:")
total_rows = check_df.count()
print(f"Tổng cộng có: {total_rows:,} dòng.")
# (Ghi chú: Số dòng này chắc chắn sẽ NHIỀU HƠN số dòng của 63 file gốc cộng lại,
# vì các ca sĩ feat với nhau đã bị lệnh explode tách thành nhiều dòng riêng biệt).

# 3. SOI LẠI CÁC "TỘI PHẠM" CŨ BẰNG MẮT THƯỜNG
print("\n🔍 3. NGHIỆM THU CÁC CA KHÓ (Alan Walker, Jimin, Imagine Dragons...):")
check_df.filter(F.col("artist_name").rlike("alan walker|yuqi|jvke|jimin|gash|imagine")) \
        .select("artist_name") \
        .distinct() \
        .show(30, truncate=False)

# 4. KIỂM TRA XEM TÊN BÀI HÁT / ALBUM ĐÃ ĐƯỢC DỊCH SANG LATIN CHƯA
print("\n🔍 4. KIỂM TRA TÊN BÀI HÁT VÀ ALBUM (Lấy ngẫu nhiên 20 dòng):")
check_df.select("artist_name", "track_name", "release_name").sample(fraction=0.001).show(20, truncate=False)

# 5. RÀ SOÁT KÝ TỰ RÁC CUỐI CÙNG
print("\n🔍 5. TÌM XEM CÒN KÝ TỰ LẠ KHÔNG (Nếu bảng dưới đây TRỐNG là SẠCH 100%):")
check_df.filter(
    F.col("artist_name").rlike(r"[^a-z0-9\s]")
).select("artist_name", "track_name").show(20, truncate=False)